# Final composed Brunelli model — independent benchmark export

This notebook simulates **only the final composed Brunelli configuration used for the benchmark**.

It does not execute the original Milan/Coppola model. The original Milan benchmark is kept in notebook 01.

The Brunelli run uses the retained site-specific initial state and pest parameters together with the final **SLA-based crop-damage interface**. The pest resource definition itself remains the retained Brunelli/Coppola resource formulation. The reference thesis notebook is included separately for provenance.


In [1]:
from pathlib import Path
import math
import numpy as np
import pandas as pd
from scipy.stats import pearsonr

HERE=Path.cwd().resolve()
DATASET=HERE/"brunelli_final_dataset.csv"
OUT=HERE/"composed_outputs"
OUT.mkdir(exist_ok=True)
assert DATASET.exists()

raw=pd.read_csv(DATASET,parse_dates=["datetime"]).sort_values("datetime").reset_index(drop=True)
start=raw["datetime"].min()
raw["Day_float"]=(raw["datetime"]-start).dt.total_seconds()/86400.0
raw["Day"]=np.floor(raw["Day_float"]).astype(int)
N_HOURS=70*24
raw=raw.iloc[:N_HOURS].copy()
hourly_temperature=raw["Temperature"].to_numpy(float)
daily_temperature=(raw.groupby("Day")["Temperature"].mean()
                   .reindex(range(70)).interpolate().ffill().bfill().to_numpy(float))

EGG_LARVA_DAYS=np.array([0,6,13,20,27,34,41,48,55,62,69],dtype=int)
ADULT_DAYS=np.array([0,7,14,21,28,36,42,48,55,62,69],dtype=int)

NORMALIZATION=False
AMPLITUDE_SCALING=False
SMOOTHING=False
POSTHOC_SHIFT=False
TRAP_Q=False


In [2]:
FINAL={
    "mu_E":0.32,"mu_L":0.22,"mu_P":0.04,"mu_A":0.27,
    "zeta_A":10.28,
    "a2":48.0,"b2":0.11,
    "a3":73.78,"b3":0.09,
    "fertility_a":290.58,"fertility_b":1.57,"fertility_c":20.64,
    "T_pest_min":4.90,"T_pest_max":32.7,
    "alpha":0.00065,"delta":35.0,
    "epsilon_max":0.55,"chi":0.004,
    "Omega":1.0/35.0,"nu":0.000745,
}
FINAL_IC={"E0":531.0,"L0":438.0,"P0":349.0,"A0":0.0,"LAI0":0.5}
DAMAGE_SLA=0.018

pd.DataFrame(
    [{"Parameter":k,"Value":v} for k,v in FINAL.items()]
    + [{"Parameter":k,"Value":v} for k,v in FINAL_IC.items()]
    + [{"Parameter":"Damage_SLA","Value":DAMAGE_SLA}]
).to_csv(OUT/"composed_run_configuration.csv",index=False)


In [3]:
def total_fertility(T,p=FINAL):
    T=float(T)
    if not (p["T_pest_min"] < T < p["T_pest_max"]):
        return 0.0
    return (p["fertility_a"]*(T-p["T_pest_min"])**p["fertility_b"]*(p["T_pest_max"]-T)
            /((p["fertility_c"]-p["T_pest_min"])**p["fertility_b"]
              *(p["T_pest_max"]-p["T_pest_min"])))

def phi(T,p=FINAL):
    return total_fertility(T,p)/(2.0*p["zeta_A"])

def delta_nodes_from_temperature(T):
    return max(0.0,0.036*(T-9.0))

nodes=22.0
TOMGRO_DLAI=[]
for T in daily_temperature:
    DN=delta_nodes_from_temperature(T)
    nodes += DN
    a,b,c,d=0.001,250000.0,1.435,3.6
    dlai=(b*d*(c-a)*nodes**(d-1)/(b+nodes**d)**2)*DN
    TOMGRO_DLAI.append(float(dlai))
TOMGRO_DLAI=np.asarray(TOMGRO_DLAI)

LARVA_SD6={"P":0.1125,"T0_K":298.8563,"HA":9683.435,"HH":2555621.0,"TH_K":310.4786,"R":1.987}
def larval_development_rate(T_c):
    q=LARVA_SD6
    T_k=float(T_c)+273.15
    numerator=q["P"]*(T_k/q["T0_K"])*math.exp((q["HA"]/q["R"])*(1.0/q["T0_K"]-1.0/T_k))
    high_inactivation=math.exp((q["HH"]/q["R"])*(1.0/q["TH_K"]-1.0/T_k))
    return max(float(numerator/(1.0+high_inactivation)),1e-12)
R25=larval_development_rate(25.0)


In [4]:
def rhs(state,T,DLAI,p=FINAL):
    E,L,P,A,LAI=np.maximum(np.asarray(state,dtype=float),0.0)
    F=LAI/p["Omega"] if p["Omega"]>0 else 0.0
    feeding=p["alpha"]*F/(F+p["delta"]) if F>0 else 0.0
    zL_resource=p["chi"]/(p["epsilon_max"]*feeding) if feeding>0 else 1e12
    zL=zL_resource*(R25/larval_development_rate(T))
    zE=p["a2"]*math.exp(-p["b2"]*T)
    zP=p["a3"]*math.exp(-p["b3"]*T)
    hatch=E/zE
    capacity=F*p["Omega"]/p["nu"]
    larval_inflow=min(hatch,max(0.0,capacity-L))
    pupation=L/zL
    emergence=P/zP
    # Crop damage is converted through the Brunelli SLA-based leaf-biomass interface.
    # The pest-resource quantity F above remains defined with Omega.
    B_leaf=LAI/DAMAGE_SLA if DAMAGE_SLA>0 else 0.0
    p_damage=p["alpha"]*B_leaf/(B_leaf+p["delta"]) if B_leaf>0 else 0.0
    damage_lai=DAMAGE_SLA*p_damage*L
    return np.array([
        phi(T,p)*A-hatch-p["mu_E"]*E,
        larval_inflow-pupation-p["mu_L"]*L,
        pupation-emergence-p["mu_P"]*P,
        emergence-p["mu_A"]*A,
        DLAI-damage_lai,
    ],dtype=float)

def simulate():
    dt=1.0/24.0
    state=np.array([FINAL_IC[k] for k in ["E0","L0","P0","A0","LAI0"]],dtype=float)
    states=[]
    cumulative=[]
    cumA=0.0
    for h,T in enumerate(hourly_temperature):
        states.append(state.copy())
        cumulative.append(cumA)
        DLAI=TOMGRO_DLAI[h//24]
        k1=rhs(state,T,DLAI)
        k2=rhs(np.maximum(state+0.5*dt*k1,0.0),T,DLAI)
        k3=rhs(np.maximum(state+0.5*dt*k2,0.0),T,DLAI)
        k4=rhs(np.maximum(state+dt*k3,0.0),T,DLAI)
        state=np.maximum(state+dt*(k1+2*k2+2*k3+k4)/6.0,0.0)
        cumA += state[3]*dt
    return np.asarray(states),np.asarray(cumulative)

hourly_states,cumulative=simulate()

hourly_df=pd.DataFrame(hourly_states,columns=["Egg","Larva","Pupa","Adult","LAI"])
hourly_df.insert(0,"Hour",np.arange(len(hourly_df)))
hourly_df["Day_float"]=hourly_df["Hour"]/24.0
hourly_df["Cumulative_Adult_Exposure_at_hour_start"]=cumulative
hourly_df.to_csv(OUT/"composed_hourly_states.csv",index=False)

daily=hourly_states[np.arange(0,N_HOURS,24)]
daily_df=pd.DataFrame(daily,columns=["Egg","Larva","Pupa","Adult","LAI"])
daily_df.insert(0,"Day",np.arange(len(daily_df)))
daily_df.to_csv(OUT/"composed_daily_states.csv",index=False)

survey=pd.DataFrame({
    "Day":EGG_LARVA_DAYS,
    "Composed_Egg":daily[EGG_LARVA_DAYS,0],
    "Composed_Larva":daily[EGG_LARVA_DAYS,1],
})
survey.to_csv(OUT/"composed_egg_larva_survey_predictions.csv",index=False)

adult_interval=np.diff(cumulative[ADULT_DAYS*24])
adult=pd.DataFrame({
    "Interval_start_day":ADULT_DAYS[:-1],
    "Interval_end_day":ADULT_DAYS[1:],
    "Composed_Adult_interval_exposure":adult_interval,
})
adult.to_csv(OUT/"composed_adult_interval_predictions.csv",index=False)

display(survey)
display(adult)


,Day,Composed_Egg,Composed_Larva
0,0,531.000000,438.000000
1,6,469.420454,628.380223
2,13,760.027945,699.625113
3,20,618.659446,699.742981
4,27,431.721241,762.935325
5,34,662.154508,878.142349
6,41,892.396602,914.069059
7,48,834.046607,907.529953
8,55,807.335569,807.754948
9,62,788.245916,828.511169


,Interval_start_day,Interval_end_day,Composed_Adult_interval_exposure
0,0,7,569.405182
1,7,14,633.333031
2,14,21,494.525859
3,21,28,596.532479
4,28,36,877.801448
5,36,42,586.234364
6,42,48,578.572938
7,48,55,668.701590
8,55,62,618.317443
9,62,69,621.133306


In [5]:
daily_first=raw.groupby("Day",as_index=False).agg(
    Eggs=("Eggs","first"),Larvae=("Larvae","first"),Adults=("Adults_Brunelli","first"))
egg_obs=daily_first.set_index("Day").loc[EGG_LARVA_DAYS,"Eggs"].to_numpy(float)
larva_obs=daily_first.set_index("Day").loc[EGG_LARVA_DAYS,"Larvae"].to_numpy(float)
adult_obs=daily_first.set_index("Day").loc[ADULT_DAYS,"Adults"].to_numpy(float)

egg_pred=survey["Composed_Egg"].to_numpy(float)
larva_pred=survey["Composed_Larva"].to_numpy(float)
adult_pred=adult["Composed_Adult_interval_exposure"].to_numpy(float)

def direction_count(obs,pred):
    return int(np.sum(np.sign(np.diff(obs))==np.sign(np.diff(pred))))

assert abs(pearsonr(egg_obs,egg_pred)[0]-0.7838305819274596)<1e-12
assert abs(pearsonr(larva_obs,larva_pred)[0]-0.7779308840693925)<1e-12
assert abs(pearsonr(adult_obs[1:],adult_pred)[0]-0.7386430200035545)<1e-12
assert direction_count(egg_obs,egg_pred)==8
assert direction_count(larva_obs,larva_pred)==8
assert direction_count(adult_obs[1:],adult_pred)==7
assert EGG_LARVA_DAYS[np.argmax(egg_pred)]==41
assert EGG_LARVA_DAYS[np.argmax(larva_pred)]==41
assert ADULT_DAYS[1:][np.argmax(adult_pred)]==36
assert DAMAGE_SLA==0.018
assert not any([NORMALIZATION,AMPLITUDE_SCALING,SMOOTHING,POSTHOC_SHIFT,TRAP_Q])
print("FINAL COMPOSED NOTEBOOK AUDIT PASSED")


FINAL COMPOSED NOTEBOOK AUDIT PASSED
